# SRA シナプス削除デモ
## — LLMから特定の知識だけを物理的に抜き取る！

このノートブックは、記事『[【AIロマン】LLMから特定の知識だけを物理的に抜き取る！Hotswappable LLMの「シナプス削除」と「パージ」](https://qiita.com/JunSuzukiJapan/items/)』のインタラクティブなデモです。

SRA（Synaptic Routing Architecture）では、不要になった知識やシナプスを2つの方法で削除できます。

| 手法 | API | 用途 |
|------|-----|------|
| **物理的な取り外し** | `pop_synapses(N)` | Hot-Swapで追加したシナプスを末尾から削除し、モデルサイズを復元 |
| **ゼロクリア（パージ）** | `clear_synapses([idx])` | 途中のインデックスのシナプスを無効化し、空きスロットに変換 |

さらに、ゼロクリア時に発生する**コサイン類似度の罠**と、その解決策も実際に確認します。

最後に、実際にマルチタスク学習済みモデルに対して `clear_synapses` を適用し、**狙ったタスクの機能だけが破壊され、他のタスクは完全に無傷（Zero Forgetting）**であることを実証します。

---
## Part 1：シナプスの取り外し (`pop_synapses`)

Hot-Swapで後から追加したシナプスを、末尾から物理的に切り落とします。
OSのプラグインをアンインストールするように、AIの脳のパーツを物理的に取り外すことができます。

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# 小さなモデルを作って、追加→取り外しの流れを確認
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== アプローチ①：シナプスの取り外し（物理的カット） ===')
print(f'[初期状態] シナプス数: {model.blocks[0].num_synapses}')
print(f'  ルーター埋め込みの形状: {model.blocks[0].router.get_full_emb().shape}')

# Hot-Swapでシナプスを2つ追加
model.add_synapses(2, freeze_base=True)
print(f'\n[追加後] シナプス数: {model.blocks[0].num_synapses}')
print(f'  ルーター埋め込みの形状: {model.blocks[0].router.get_full_emb().shape}')

# 追加したシナプスを物理的に取り外し
model.pop_synapses(2)
print(f'\n[取り外し後] シナプス数: {model.blocks[0].num_synapses}')
print(f'  ルーター埋め込みの形状: {model.blocks[0].router.get_full_emb().shape}')
print('\n✅ メモリ使用量が完全に復元されました！')


---
## Part 2：ゼロクリア（パージ）と「コサイン類似度の罠」

途中のインデックスのシナプスを削除したい場合、物理的に削除するとIDがズレてしまいます。
そこで、重みを `0.0` で上書きする「ゼロクリア」を行います。

しかし、ここには**恐ろしい罠**があります。ゼロベクトルのコサイン類似度は `0.0` になり、
正常なシナプスのスコアが負の場合、**空っぽのシナプスの方がスコアが高くなってデータが吸い込まれる**という
ブラックホール現象が発生します。

SRAのルーターには、この罠を防ぐ **`-inf` マスク** が組み込まれています。実際に確認してみましょう。

In [ ]:
print('=== アプローチ②：ゼロクリアと -inf マスクの動作確認 ===')

# 新しいモデルを作成
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# ゼロクリア前のシナプス2番の重みを確認
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[ゼロクリア前] シナプス {target_idx} のルーター埋め込みノルム: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 ノルム: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# ゼロクリア実行
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[ゼロクリア後] シナプス {target_idx} のルーター埋め込みノルム: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 ノルム: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\n💥 シナプス2番の重みが完全にゼロになりました！')


In [ ]:
# -inf マスクの動作を実際に確認
print('=== -inf マスクの動作確認 ===')

# ダミー入力でルーターを実行
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# ルーターのlogitsを確認（最終レイヤー）
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'ルーターのスコア（1トークン目）:')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = '🔒 封鎖 (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' ← ゼロクリア済み' if i == target_idx else ''
    print(f'  シナプス {i}: {label}{marker}')

print('\n✅ ゼロクリアされたシナプスのスコアが -inf に設定され、')
print('   ルーターは絶対にこのシナプスを選ばなくなりました。')
print('   これにより「ブラックホール現象」が完全に防止されています。')


---
## Part 3：機能的な証明 — 学習済みモデルでの `clear_synapses`

ここからが本番です。Part 1, 2 では「APIの動作」を確認しましたが、
本当に重要なのは **「ゼロクリアした後、削除した知識だけが失われ、他の知識は完全に無傷なのか？」** という機能的な証明です。

ノートブック05（Lesion実験）と同じ手法で：
1. `copy` と `reverse` をマルチタスク学習させる
2. `reverse` で頻繁に使われているシナプスを特定し、`clear_synapses` で削除
3. `reverse` は解けなくなるが、`copy` は100%正解し続ける（Zero Forgetting）ことを確認

In [ ]:
# ノートブック05と完全に同じ設定
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# マルチタスク学習（ノートブック05と同じ）
print('学習開始... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('学習完了！')


### 3-1. 削除前の性能テストとターゲットの特定
各タスクが正しく解けることを確認し、`reverse` で最も使われているシナプスを特定します。

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # 使われたシナプスの集計 (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] 正解率: {acc*100:.1f}%')
    return usage

print('=== 削除前 (Before Deletion) ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Reverse で多く使われ、Copy ではあまり使われていないシナプスを特定
# (ノートブック05と同じロジック)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# もし完全に分離できていない場合は、Reverseで最も使われたシナプスを1つ選ぶ
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> 削除ターゲット: シナプス {target_synapses} (REVERSEで突出して使用)')


### 3-2. `clear_synapses` によるシナプスの削除
ノートブック05では手動で `block.w1.data[idx].zero_()` を行いましたが、
ここでは**正式な `clear_synapses` API** を使って、ルーターの `-inf` マスクも含めた
安全な削除を行います。

In [ ]:
print(f'シナプス {target_synapses} を clear_synapses で削除します...')

# 削除前のノルムを記録
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [削除前] シナプス {idx}: Router埋込ノルム={emb_norm:.4f}, W1ノルム={w1_norm:.4f}')

# clear_synapses API を使用（ルーターの -inf マスクも自動適用）
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\n💥 削除完了！')

# 削除後のノルムを確認
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [削除後] シナプス {idx}: Router埋込ノルム={emb_norm:.4f}, W1ノルム={w1_norm:.4f}')


### 3-3. 削除後の性能テスト（Zero Forgettingの検証）

一部のシナプスが削除された状態で、再度テストを行います。

**期待される結果:**
- **Copy**: 正解率が維持（✅ 別のシナプスを使っているため、完全に無傷）
- **Reverse**: 正解率が低下（❌ 専門シナプスが失われたため）

In [ ]:
print('=== 削除後 (After Deletion) ===')
test_task('copy')
test_task('reverse')

print('\n💡 【考察】')
print('1つの大きなニューラルネットの一部をゼロで破壊すると、')
print('通常は全てのタスクが壊滅します。')
print('しかしSRAでは、タスクごとに独立した専門家経路（シナプス）が')
print('自律的に形成されているため、clear_synapses で特定のシナプスを削除しても、')
print('別のシナプスを使っているタスクには一切影響が出ません。')
print('\nこれこそが、モジュラーAIとしてのSRAの真の強みです。')
print('削除されたシナプスの空きスロットには、後から新しいプラグインを')
print('上書き（Hot-Swap）して再利用することも可能です！')


---
## まとめ

| デモ | 実証した内容 |
|------|-------------|
| Part 1 | `pop_synapses` による物理的な取り外しとメモリ復元 |
| Part 2 | `clear_synapses` によるゼロクリアと `-inf` マスクの動作確認 |
| Part 3 | 学習済みモデルでの `clear_synapses` → 狙ったタスクだけ破壊、他は無傷 |

これにて、**「学習 → 合体（Hot-Swap） → 削除（パージ） → スロットの再利用（上書き）」**という
モジュラーAIとしての完全なライフサイクルが実現できました。